In [58]:
import requests
from requests.structures import CaseInsensitiveDict
import numpy_financial as npf
import numpy as np
from datetime import datetime as dt
import pandas as pd

headers = CaseInsensitiveDict()
headers["accept"] = "application/json"
baseUrlApi = "https://fintual.cl/api/real_assets/"

class Portfolio:

    def __init__(self, id, doc = True):
        urlAssetInfo         = f"{baseUrlApi}{id}"
        assetInfo            = requests.get(urlAssetInfo,  headers=headers).json()['data']['attributes']

        self.id          = id
        self.name        = assetInfo['name']
        self.startDate   = assetInfo['start_date'] 
        self.lastDate    = pd.to_datetime(assetInfo['last_day']['date'], format='%Y-%m-%d')
        # self.lastDate    = assetInfo['last_day']['date'] 
        self.lastPrice   = assetInfo['last_day']["net_asset_value"]
        # self.df          = pd.DataFrame() 
        # self.df          = self.get_all_days()

        # Url de la api de funtual necesario para obtener todos los valores existentes en el tiempo
        
        

        if doc == True:
            # Revisar y actualizar datos
            days = f"?from_date=2024-12-22"
            pass
        else:
            # Crear archivo
            days = f""
            pass


        urlAssetInfoDays  = f"{baseUrlApi}{id}/days{days}?to_date=2024-08-28" #?to_date=2024-08-28
        # urlAssetInfoDays  = f"{baseUrlApi}{id}/days" #?to_date=2024-08-28
        # https://fintual.cl/api/real_assets/186/days?from_date=2024-12-20

        # obtenemos los datos de la api y lo pasamos a un DataFrame
        assetInfoDays        = requests.get(urlAssetInfoDays,  headers=headers).json()['data']
        df = pd.DataFrame(assetInfoDays)

        # Nomalizamos los datos a Json
        df = pd.json_normalize(df['attributes'])[['date', 'price', 'shareholders', 'total_assets', 'total_net_assets', 'outstanding_shares']]
        # En caso que los datos del primer día aún no estén disponibles dejamos dicha fila fuera del DataFrame
        # if pd.Series(np.isnan(df.tail(1)['total_assets']) == True).all():
        #     df = df[1:-1]

        # Elimina las filas las cuales no dispongan del datos clave 'total_assets'
        df.dropna(subset=['total_assets'], inplace=True)

        # Se cambia el formato de la columna fecha para generar filtros y crear nuevas columnas con los años, mese y día
        df['date']  = pd.to_datetime(df['date'],format='%Y-%m-%d')
        df['year']  = df['date'].dt.year
        df['month'] = df['date'].dt.month
        df['day']   = df['date'].dt.day
        
        # Cambio de nombre de las columas 
        df.columns = ('fecha','precio','accionistas','activos_totales','activos_neto_totales','acciones_en_circulación', 'año', 'mes', 'día')

        df = df[::-1].reset_index(drop=True)

        self.df = df 

        
    def __str__(self):
        return self.name

In [2]:
import pandas as pd

archivo_excel = "portafolio.xlsx"
id_portfolio = [186,187,188, 15077]

try:
    # La información se actualizará 
    pd.ExcelFile(archivo_excel)    


    # info = pd.read_excel(f"{archivo_excel}",sheet_name='Risky Norris')

    print(f"El archivo '{archivo_excel}' existe y es un archivo Excel válido.")

except FileNotFoundError:
    # Se creará el documento
    print(f"El archivo '{archivo_excel}' no existe.")
    li_port = []
    for n in id_portfolio:
        li_port.append(Portfolio(n))

    with pd.ExcelWriter("precios.xlsx") as writer:
        for port_n in li_port:
            port_n.df.to_excel(writer, sheet_name=f"{port_n.name}") 

except Exception as e:
    #Se mostrará el error
    print(f"Ocurrió un error al intentar leer el archivo: {e}")

El archivo 'portafolio.xlsx' no existe.


In [48]:
port186 = Portfolio(186)
# port186.get_all_days()
dt186 = port186.df

In [24]:
dt186

,fecha,precio,accionistas,activos_totales,activos_neto_totales,acciones_en_circulación,año,mes,día
0,2018-02-13,1003.8325,1.0,7.083570e+05,4.015330e+05,4.000000e+02,2018,2,13
1,2018-02-14,1013.2619,2.0,1.291954e+06,9.853050e+05,9.724088e+02,2018,2,14
2,2018-02-15,1016.1704,2.0,1.133173e+06,1.133133e+06,1.115101e+03,2018,2,15
3,2018-02-16,1016.2871,2.0,1.133340e+06,1.133263e+06,1.115101e+03,2018,2,16
4,2018-02-17,1016.2728,2.0,1.133361e+06,1.133247e+06,1.115101e+03,2018,2,17
...,...,...,...,...,...,...,...,...,...
2503,2024-12-21,3030.0167,56670.0,4.056040e+11,2.927978e+11,9.663242e+07,2024,12,21
2504,2024-12-22,3029.9200,56670.0,4.056042e+11,2.927885e+11,9.663242e+07,2024,12,22
2505,2024-12-23,3060.0264,56752.0,4.059942e+11,2.965236e+11,9.690229e+07,2024,12,23
2506,2024-12-24,3083.3180,56789.0,4.093986e+11,2.992437e+11,9.705250e+07,2024,12,24


In [52]:
print(dt186.iloc[-1]['fecha'])        
print(port186.lastDate)

print(type(dt186.iloc[-1]['fecha']))  
print(type(port186.lastDate))

dt186.iloc[-1]['fecha'] <= port186.lastDate

2024-12-22 00:00:00
2024-12-25 00:00:00
<class 'pandas._libs.tslibs.timestamps.Timestamp'>
<class 'pandas._libs.tslibs.timestamps.Timestamp'>


True

In [57]:
dt186[dt186['fecha'] < dt186.iloc[-1]['fecha']]
# dt186[dt186['fecha'] < port186.lastDate]

,fecha,precio,accionistas,activos_totales,activos_neto_totales,acciones_en_circulación,año,mes,día
0,2018-02-13,1003.8325,1.0,7.083570e+05,4.015330e+05,4.000000e+02,2018,2,13
1,2018-02-14,1013.2619,2.0,1.291954e+06,9.853050e+05,9.724088e+02,2018,2,14
2,2018-02-15,1016.1704,2.0,1.133173e+06,1.133133e+06,1.115101e+03,2018,2,15
3,2018-02-16,1016.2871,2.0,1.133340e+06,1.133263e+06,1.115101e+03,2018,2,16
4,2018-02-17,1016.2728,2.0,1.133361e+06,1.133247e+06,1.115101e+03,2018,2,17
...,...,...,...,...,...,...,...,...,...
2499,2024-12-17,3108.8840,56591.0,4.080803e+11,2.991241e+11,9.621590e+07,2024,12,17
2500,2024-12-18,2997.9504,56620.0,4.045800e+11,2.886963e+11,9.629788e+07,2024,12,18
2501,2024-12-19,3013.5461,56628.0,3.974446e+11,2.906536e+11,9.644903e+07,2024,12,19
2502,2024-12-20,3030.1134,56670.0,4.056037e+11,2.928072e+11,9.663242e+07,2024,12,20


In [69]:
baseUrlApi = "https://fintual.cl/api/real_assets/"
id=186
url_info  = f"{baseUrlApi}{id}/days?from_date={str(dt186.iloc[-1]['fecha'])[:10]}" #?to_date=2024-08-28
        # url_info  = f"{baseUrlApi}{id}/days" #?to_date=2024-08-28
        # https://fintual.cl/api/real_assets/186/days?from_date=2024-12-20

        # obtenemos los datos de la api y lo pasamos a un DataFrame
info_day_1        = requests.get(url_info,  headers=headers).json()['data']
df2 = pd.DataFrame(info_day_1)
df2

,id,type,attributes
0,186-2024-12-22,real_asset_day,"{'date': '2024-12-22', 'price': 3029.92, 'fixe..."
1,186-2024-12-23,real_asset_day,"{'date': '2024-12-23', 'price': 3060.0264, 'fi..."
2,186-2024-12-24,real_asset_day,"{'date': '2024-12-24', 'price': 3083.318, 'fix..."
3,186-2024-12-25,real_asset_day,"{'date': '2024-12-25', 'price': 3083.2207, 'fi..."


In [70]:
urlAssetInfoDays2 = f"{baseUrlApi}{id}/days?to_date=2024-12-22" #?to_date=2024-08-28
assetInfoDays2        = requests.get(urlAssetInfoDays2,  headers=headers).json()['data']
ds = pd.DataFrame(assetInfoDays2)
ds

,id,type,attributes
0,186-2024-12-22,real_asset_day,"{'date': '2024-12-22', 'price': 3029.92, 'fixe..."
1,186-2024-12-21,real_asset_day,"{'date': '2024-12-21', 'price': 3030.0167, 'fi..."
2,186-2024-12-20,real_asset_day,"{'date': '2024-12-20', 'price': 3030.1134, 'fi..."
3,186-2024-12-19,real_asset_day,"{'date': '2024-12-19', 'price': 3013.5461, 'fi..."
4,186-2024-12-18,real_asset_day,"{'date': '2024-12-18', 'price': 2997.9504, 'fi..."
...,...,...,...
2501,186-2018-02-16,real_asset_day,"{'date': '2018-02-16', 'price': 1016.2871, 'fi..."
2502,186-2018-02-15,real_asset_day,"{'date': '2018-02-15', 'price': 1016.1704, 'fi..."
2503,186-2018-02-14,real_asset_day,"{'date': '2018-02-14', 'price': 1013.2619, 'fi..."
2504,186-2018-02-13,real_asset_day,"{'date': '2018-02-13', 'price': 1003.8325, 'fi..."


In [71]:
info = pd.read_excel(f"precios.xlsx",sheet_name='Risky Norris')
pd.ExcelFile()

FileNotFoundError: [Errno 2] No such file or directory: 'precios.xlsx'